In [2]:
notebook_path_offset = "../../"

In [4]:
import os
import sys
sys.path.append(
    os.path.join(
        notebook_path_offset,
        "src"
    )
)

In [5]:
import numpy as np
import scipy.stats
from typing import List

import jaccard_index_distribution

### **Configuring the problem**

There are two boxes, each contains $N$ balls (=classes). We take $n$ balls from each box and estimate the Jaccard index as a function of intersection $k$, $f(k) = k / (2n - k)$. We repeat the procedure $s\cdot N$ times, with $s$=number of samples per class, and compute the mean Jaccard value. For the outer loop, we repeat the above computations $n_{\text{trials}}$ times in order to estimate quantiles, inter-quantile range and a higher outlier threshold.

In [6]:
# Imagenet ILSVRC, validation partition + top-5

N: int = 1000 # number of classes
s: int = 50 # number of samples per class
n: int = 5 # top-n

n_trials: int = 100 # number of repetitions for statistical evaluation

### **Distribution of mean Jaccard indices**

Manual computations:

In [7]:
psum: float = 0.
expected_jval: float = 0.
for k in range(n+1):
    pval: float = jaccard_index_distribution.compute_proba(
        N, n, k
    )
    jval: float = jaccard_index_distribution.compute_jac_outcome(
        k, n
    )
    print(f"k={k}: prob={pval:.3e}  jval={jval:.5f}")

    psum += pval
    expected_jval += pval*jval

assert np.isclose(psum, 1.)
print(f"Expected mean jaccard index value: {expected_jval:.5f}")

k=0: prob=9.752e-01  jval=0.00000
k=1: prob=2.460e-02  jval=0.11111
k=2: prob=1.984e-04  jval=0.25000
k=3: prob=5.994e-07  jval=0.42857
k=4: prob=6.030e-10  jval=0.66667
k=5: prob=1.212e-13  jval=1.00000
Expected mean jaccard index value: 0.00278


Using scipy.stats:

In [8]:
rand_variable: scipy.stats._distn_infrastructure.rv_frozen = (
    scipy.stats.hypergeom(N, n, n)
)
k_values: np.ndarray = np.arange(n+1)
pmf_values: np.ndarray = rand_variable.pmf(k_values)

In [9]:
psum: float = 0.
expected_jval: float = 0.
for k in range(n+1):
    pval: float = pmf_values[k]
    jval: float = jaccard_index_distribution.compute_jac_outcome(
        k, n
    )
    print(f"k={k}: prob={pval:.3e}  jval={jval:.5f}")

    psum += pval
    expected_jval += pval*jval

assert np.isclose(psum, 1.)
print(f"Expected mean jaccard index value: {expected_jval:.5f}")

k=0: prob=9.752e-01  jval=0.00000
k=1: prob=2.460e-02  jval=0.11111
k=2: prob=1.984e-04  jval=0.25000
k=3: prob=5.994e-07  jval=0.42857
k=4: prob=6.030e-10  jval=0.66667
k=5: prob=1.212e-13  jval=1.00000
Expected mean jaccard index value: 0.00278


### **Full simulation of the procedure**

On each trial, we manually sample top-n reference vector and $N*s$ top-n samples to be matched with it.

In [10]:
mean_jac_vals_simulated_list: List[float] = (
    jaccard_index_distribution.two_boxes_full_simulation(
        N, s, n, n_trials, verbose=True
    )
) 

In [11]:
tau_simulated: float = (
    jaccard_index_distribution.compute_hiqr_outlier_threshold(
        mean_jac_vals_simulated_list
    )
)
print(
    f"N={N}, s={s}, n={n}: tau = {tau_simulated:.5f}"
    f" [simulated; n_trials = {n_trials}]"
)

N=1000, s=50, n=5: tau = 0.00297 [simulated; n_trials = 100]


### **Sampling shortcut**

Using hypergeometric distribution

In [16]:
mean_jac_vals_sampled_list: List[float] = (
    jaccard_index_distribution.two_boxes_distribution_sampling(
        N, s, n, n_trials, verbose=True, verbose_stats=True
    )
) 

In [17]:
tau_sampled: float = (
    jaccard_index_distribution.compute_hiqr_outlier_threshold(
        mean_jac_vals_sampled_list
    )
)
print(
    f"N={N}, s={s}, n={n}: tau = {tau_sampled:.5f}"
    f" [sampled; n_trials = {n_trials}]"
)

N=1000, s=50, n=5: tau = 0.00296 [sampled; n_trials = 100]


### **Final estimation of a higher outlier threshold**

In [51]:
n_trials_final: int = 10_000_000 # 10^7

mean_jac_vals_sampled_list2: List[float] = (
    jaccard_index_distribution.two_boxes_distribution_sampling(
        N, s, n, n_trials=n_trials_final, verbose=True,
        verbose_stats=True
    )
) 

In [72]:
tau_sampled_final: float = (
    jaccard_index_distribution.compute_hiqr_outlier_threshold(
        mean_jac_vals_sampled_list2
    )
)
print(
    f"N={N}, s={s}, n={n}: tau = {tau_sampled_final:.5f}"
    f" [sampled; n_trials = {n_trials_final}]"
)

N=1000, s=50, n=5: tau = 0.00300 [sampled; n_trials = 10000000]
